In [0]:
import requests
from urllib.parse import urljoin
from pathlib import Path
import re
import os

url = "https://download.bls.gov/pub/time.series/pr/"
volume_path  = "/Volumes/workspace/default/rearctest"

response = requests.get(
    url,
    timeout=60,
    headers={
        "User-Agent": "my-databricks-ingestion/1.0"
    }
)
response.raise_for_status()
response_text = response.text
links = re.findall(r'href=["\']([^"\']+)["\']', response_text, re.IGNORECASE)

for link in links:
    file_url = urljoin(url, link)
    filename = Path(link).name

    if not filename:
        continue

    output_path = f"{volume_path}/{filename}"
    # Skip files that have already been ingested
    if os.path.exists(output_path):
        print(f"Already exists, skipping: {filename}")
        continue

    print(f"Downloading {file_url}")

    with requests.get(file_url, 
                      stream=True, 
                      timeout=120,
                      headers={"User-Agent": "my-databricks-ingestion/1.0"}) as response:
        response.raise_for_status()

        with open(output_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    print(f"Saved: {output_path}")

In [0]:
import json

response = requests.get(
    "https://honolulu-api.datausa.io/tesseract/data.jsonrecords?cube=acs_yg_total_population_1&drilldowns=Year%2CNation&locale=en&measures=Population",
    timeout=60)
response.raise_for_status()
data = response.json()
output_path = f"{volume_path}/population.json"

with open(output_path, "w") as f:
    json.dump(data, f)

print(f"Saved to {output_path}")